# Load Cached AnnData Objects

This notebook is now for working with the cached `.h5ad` files only. It does not convert Seurat `.rds` files and it should not remake AnnData files during normal analysis.

Current cached files:

- Shi 2019 paper QC: `$PROJECT_ROOT/results/python_anndata/shi_2019_paper_qc.h5ad`
- Varela DIV30: `$PROJECT_ROOT/results/python_anndata/varela_div30.h5ad`
- Varela DIV90: `$PROJECT_ROOT/results/python_anndata/varela_div90.h5ad`

If a source Seurat object changes, rebuild the cache with the Slurm template from a login-node terminal:

```bash
cd /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline
cp slurm_templates/08_convert_python_anndata.sbatch.template   /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/jobs/08_convert_python_anndata.sbatch
sbatch /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/jobs/08_convert_python_anndata.sbatch
```

The Slurm job skips existing current `.h5ad` files unless the template is changed to run with overwrite.

## Kernel And Path Setup

Use the `Python (mge-organoid-python)` kernel. This cell adds the repo-local Python package under `python_notebooks/src` so imports work from the notebook without installing the package into site-packages.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
if len(cwd.parents) >= 2:
    candidate_roots.append(cwd.parents[1])

for root in candidate_roots:
    src = root / "python_notebooks" / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        repo_root = root
        break
else:
    raise RuntimeError("Could not locate python_notebooks/src from the current notebook directory")

repo_root

In [ ]:
import platform
import shutil

import matplotlib.pyplot as plt
import pandas as pd

from mge_organoid_python import (
    cached_h5ad_path,
    default_studies,
    load_cached_anndatas,
    missing_cached_h5ads,
    resolve_project_root,
    validate_source_paths,
)

project_root = resolve_project_root(os.environ.get("PROJECT_ROOT"))
studies = default_studies()
studies_by_id = {study.study_id: study for study in studies}

print("PROJECT_ROOT =", project_root)
for study in studies:
    print(f"{study.study_id}: {cached_h5ad_path(study, project_root=project_root)}")

## Runtime Diagnostics

This confirms where the notebook kernel is running. Loading cached `.h5ad` files is allowed on a login node for light checks, especially with `backed="r"`. Do not run Seurat conversion from this notebook.

In [ ]:
hostname = platform.node()
print("hostname:", hostname)
print("python:", sys.executable)
print("PROJECT_ROOT:", os.environ.get("PROJECT_ROOT"))
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
print("SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID"))
print("Rscript:", shutil.which("Rscript"))

if hostname.startswith("gl-login"):
    print("NOTE: This kernel is on a login node. Keep work light and use backed='r'.")
else:
    print("NOTE: This kernel is not on a login node.")

## Confirm Inputs And Cached Outputs

The Seurat source path check tells you whether the canonical `.rds` files still exist. The cached output check is the important one for this notebook: all three `.h5ad` files must exist before analysis cells run.

In [ ]:
source_missing = validate_source_paths(studies)
if source_missing:
    raise FileNotFoundError(f"Missing Seurat source files: {source_missing}")

cache_missing = missing_cached_h5ads(studies, project_root=project_root)
if cache_missing:
    raise FileNotFoundError(
        "Missing cached H5AD files: {}. Rebuild them with "
        "slurm_templates/08_convert_python_anndata.sbatch.template before running analysis.".format(cache_missing)
    )

cache_table = pd.DataFrame(
    [
        {
            "study_id": study.study_id,
            "label": study.label,
            "h5ad_path": str(cached_h5ad_path(study, project_root=project_root)),
            "size_gb": round(cached_h5ad_path(study, project_root=project_root).stat().st_size / 1024**3, 2),
        }
        for study in studies
    ]
)
cache_table

## Load Cached AnnData

The default is `backed="r"`, which opens each `.h5ad` read-only and avoids loading the full expression matrix into memory. This is the normal mode for inspection, metadata work, and UMAP plotting.

Use `backed=None` only when you intentionally need full in-memory AnnData for downstream methods that require modifying or densifying data.

In [ ]:
adatas, reports = load_cached_anndatas(studies, project_root=project_root, backed="r")
reports_df = pd.DataFrame([report.as_dict() for report in reports])
reports_df

## UMAP Preview

These plots use `adata.obsm["X_umap"]` from the cached AnnData files. They do not touch the original Seurat objects.

In [ ]:
def plot_umap(adata, title, color_by=None, s=2, alpha=0.6):
    umap = adata.obsm["X_umap"]
    fig, ax = plt.subplots(figsize=(6, 5))
    if color_by and color_by in adata.obs:
        values = adata.obs[color_by].astype("category")
        codes = values.cat.codes
        scatter = ax.scatter(umap[:, 0], umap[:, 1], c=codes, s=s, alpha=alpha, cmap="tab20")
        ax.set_title(f"{title} colored by {color_by}")
    else:
        ax.scatter(umap[:, 0], umap[:, 1], s=s, alpha=alpha)
        ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal", adjustable="box")
    return fig, ax

plot_umap(adatas["shi_2019_paper_qc"], "Shi 2019 paper QC")
plot_umap(adatas["varela_div30"], "Varela DIV30")
plot_umap(adatas["varela_div90"], "Varela DIV90")
plt.show()

## Working With One Object

Use the `adatas` dictionary for downstream analysis. Keys are:

- `shi_2019_paper_qc`
- `varela_div30`
- `varela_div90`

In [ ]:
adata = adatas["varela_div30"]
print(adata)
print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))